# 00 · Genel bakış: FreeQuantum pipeline'ını laptopta simüle etmek

**Referans makale:** Günther et al., *How to Use Quantum Computers for Biomolecular Free Energies*,
J. Chem. Theory Comput. 2026, 22, 4329–4345 (FreeQuantum pipeline'ı).

## Makale ne yapıyor?
Bir ilacın proteine **bağlanma serbest enerjisini** ($\Delta G_\text{bağlanma}$) hesaplamak istiyoruz.
Tam kuantum mekaniksel hesap binlerce atom için imkânsız. Bu yüzden katmanlı bir model kullanılıyor:

1. **MM:** Klasik kuvvet alanı ile MD ve alkimik serbest enerji (FEP).
2. **QM/MM:** Ligand (kuantum bölgesi) DFT ile, gerisi MM ile hesaplanıyor. Bu veriyle bir ML potansiyeli (ML1) eğitiliyor.
3. **QM/QM/MM:** Ligandın içindeki küçük bir *kuantum çekirdeği* çok yüksek doğrulukla (CASCI/NEVPT2, CCSD(T)) hesaplanıyor.
   ML1, transfer öğrenme ile ML2'ye dönüştürülüyor.
4. **Kuantum bilgisayar:** Gelecekte 3. adımdaki kuantum çekirdeği enerjileri bir kuantum bilgisayarda
   (kuantum faz tahmini, QPE) hesaplanacak. Makale bunun için kaynak tahmini veriyor.

## Bu demoda ne yapıyoruz?
Aynı mantığı **küçültülmüş, laptopta çalışan** bir sistemde baştan sona simüle ediyoruz. Kullandığımız sistem
**β-siklodekstrin (konakçı) + siklopentanol (misafir)**. Bu ilaç benzeri bir "bağlanma" problemi ve
gerçek deneysel bağlanma serbest enerjisi biliniyor:

> ΔG_deney = −3.050 ± 0.010 kcal/mol = **−12.76 kJ/mol** (Rekharsky et al., *J. Phys. Chem. B* 1997, 101, 87).

Girdi dosyaları da gerçek, yayımlanmış bir benchmark setinden geliyor: MobleyLab *benchmarksets*, CD Set 2.

| Makale (FreeQuantum) | Bu demo | Bu demoda nerede? | Gerçek projede nerede? |
|---|---|---|---|
| GRP78 proteini + NKP1339 (Ru ilacı) | β-siklodekstrin + siklopentanol | — | — |
| MM-FEP (Amber ff99SB-ILDN), MBAR | MM-FEP (q4md-CD + GAFF), MBAR | laptop GPU | GPU kümesi |
| QM/MM, PBE-D3/def2-SVP, ~2000 nokta | QM/MM, PBE/def2-SVP, onlarca nokta | laptop CPU | **HPC (CPU)** |
| ML1 + aktif öğrenme + NEQ anahtarlama | Gauss süreci + aktif öğrenme + Zwanzig | laptop | GPU (ML eğitimi) |
| Kuantum çekirdek: DMRG-CASCI + NEVPT2 | CASCI(4,4) + NEVPT2 | laptop | **HPC** → gelecekte **kuantum bilgisayar** |
| ML2 (transfer öğrenme) | GP düzeltme modeli | laptop | GPU |
| QPE kaynak tahmini (Şekil 4) | QPE/VQE/Trotter simülasyonu + ölçekleme | laptop (≤ ~24 qubit) | **Hata düzeltmeli kuantum bilgisayar** |

## Notebook sırası
| # | Notebook | Makaledeki karşılığı |
|---|---|---|
| 01 | Veri ve sistem hazırlama | Şekil 2, gri kutu |
| 02 | MM serbest enerjisi (FEP + MBAR) | Bölüm 2.1, Denklem 3–5 |
| 03 | QM bölgesi ve QM/MM enerjileri | Bölüm 2.3, Denklem 6 |
| 04 | ML1 + aktif öğrenme → ΔG(MM+ML1) | Bölüm 2.2, Şekil 6–7 |
| 05 | Kuantum çekirdeği + ML2 → ΔG(MM+ML2) | Bölüm 2.3–2.4, Denklem 7 |
| 06 | Kuantum algoritmaları (VQE, QPE, Trotter) | Bölüm 3, Şekil 4d |
| 07 | Kaynak tahmini, hata yayılımı, yol haritası | Bölüm 3, Şekil 4a–c |

> **Dürüst not:** Bu bir *eğitim simülasyonu*. Amaç, her adımın ne yaptığını ve maliyetini görmek.
> Kısa örnekleme nedeniyle sayılar istatistiksel olarak gürültülü olacak. Hangi adımın büyütülünce
> HPC veya kuantum bilgisayar gerektirdiğini 07. notebook'ta ölçülen sürelerden çıkaracağız.

## Ortam kontrolü

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from qfe import config, store
from qfe.plotting import C, LEG, LEG_TR
import numpy as np, matplotlib.pyplot as plt
# Ön ayar: "test" (hızlı kontrol), "demo" (ekibe gösterim), "standart" (daha güvenilir)
config.set_preset(os.environ.get("QFE_PRESET", "demo"))
print("Aktif ön ayar:", config.preset_name())

Aktif ön ayar: test


In [2]:
import importlib, platform
print("Python", platform.python_version(), "|", platform.platform())
for m in ["openmm", "openmmtools", "pymbar", "parmed", "pyscf", "openfermion", "sklearn", "scipy", "numpy"]:
    try:
        mod = importlib.import_module(m)
        print(f"  ✓ {m:12s} {getattr(mod, '__version__', '')}")
    except Exception as e:
        print(f"  ✗ {m:12s} EKSİK -> {e}")

Python 3.13.16 | Linux-6.18.44-fc-v114-x86_64-with-glibc2.39
  ✓ openmm       8.6.1


  ✓ openmmtools  0.27.0
  ✓ pymbar       4.0.3


  ✓ parmed       4.3.1
  ✓ pyscf        2.14.0


  ✓ openfermion  1.8.1
  ✓ sklearn      1.9.1
  ✓ scipy        1.18.1
  ✓ numpy        2.5.4


In [3]:
import openmm as mm
plats = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
print("OpenMM platformları:", plats)
if "CUDA" in plats:
    print("NVIDIA GPU (CUDA) kullanılacak ✓")
elif "OpenCL" in plats:
    print("OpenCL kullanılacak (GPU olabilir)")
else:
    print("Sadece CPU var. 'test' ön ayarıyla çalışın; 'demo' çok uzun sürebilir.")

OpenMM platformları: ['Reference', 'CPU']
Sadece CPU var. 'test' ön ayarıyla çalışın; 'demo' çok uzun sürebilir.


### Termodinamik döngü (çift ayrıştırma / double decoupling)

```
  misafir(su) + konakçı(su)  ──ΔG_bağlanma──►  kompleks(su)
        │                                          │
  ΔG_s  │ misafiri "kapat"                  ΔG_c   │ misafiri "kapat" (kısıtlı)
        ▼                                          ▼
  misafir(gaz benzeri) + konakçı(su) ◄── kT·ln(V0/V_site) ── kapalı misafir konakçıda
```
ΔG_bağlanma = ΔG_s − ΔG_c + kT ln(V0/V_site). Bu, makaledeki Denklem 5'in aynısı.